# This code was run on a server that was GPU enabled using apptainer

## Apptainer setup

In [ ]:
cd PROJECT_DIRECTORY
export APPTAINER_CACHEDIR=SOME_APPTAINER_DIR
apptainer pull pytorch26.sif docker://nvcr.io/nvidia/pytorch:24.12-py3
apptainer exec pytorch26.sif python -m venv --system-site-packages env_name


In [ ]:
cd /projects/academic/kjoseph/kenny/container-directory
apptainer shell --nv pytorch26.sif
python

## Code to implement identification and masking of named entities

In [ ]:

import spacy
spacy.require_gpu()
import pandas as pd
from charai_utils import *
from tqdm import tqdm
nlp = spacy.load("en_core_web_trf", disable=["tagger", "parser", "attribute_ruler", "lemmatizer"])

df = pd.read_parquet("english_5per.parquet")

res = []
for i, row in tqdm(df.iterrows(),total=len(df)):
    res.append([row.url] + mask(row.greeting,nlp))

pd.DataFrame(res, columns=['url','greeting','ents']).to_parquet("mask_result.parquet",index=False)

## Code to generate sentence embeddings from entity-masked greetings

In [5]:
from sentence_transformers import SentenceTransformer
sentence_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", 
                                     trust_remote_code=True,
                                    cache_folder= "./st_cache")

import pandas as pd
docs = pd.read_parquet("mask_result.parquet").greeting.values.tolist()
print(len(docs))
embeddings = sentence_model.encode(docs, show_progress_bar=True)
import numpy as np
np.save("allmpv2_embeddings.npy", embeddings)


,url,greeting,ents
0,https://character.ai/chat/L54EeXsuNo2-7vOEF13d...,You were braiding his hair gently while he was...,"[conan, babe, 23 years old]"
1,https://character.ai/chat/oi6m-YU-UP9Px-nK0k6A...,"your boyfriend is bored, he's alone at the hou...","[acel, 21]"
2,https://character.ai/chat/TxUEgehqDV5ej9W2nTX7...,You're boyfriend [MASK].You guys have been a c...,"[2 years, dimitri]"
3,https://character.ai/chat/FUktRQ6xCiWlRVyHyDmO...,Your model boyfriend [MASK] invited you to go...,[aziel]
4,https://character.ai/chat/d07v_bSLvRGZqWjVHuNV...,"you're quiet boyfriend, [MASK] had been dating...","[about a year, hugo]"
...,...,...,...
95,https://character.ai/chat/O6BNjZJrDqzUwgrB694j...,You didn't give [MASK] permission when you wer...,[zeus]
96,https://character.ai/chat/1Lz7tVXY_nP3COgoPe-l...,[MASK] and you attend a mafia party. [MASK] do...,[keith]
97,https://character.ai/chat/CHw85dtkIiA-uulfRMf8...,You've been waiting since morning for [MASK] t...,"[morning, 8 am, leo]"
98,https://character.ai/chat/-1I1882T16k01j6DB7Kt...,Today you and [MASK] went for a walk in the pa...,"[today, all day, dexter]"


## Code to construct power relationships for analysis

In [1]:

def mask(doc,nlp):
    ents = set()
    doclist = [char for char in doc]
    nlpdoc = nlp(doc)

    for ent in nlpdoc.ents[::-1]:
        if ent.label_ in {"PERSON"}:
            doclist[ent.start_char : ent.end_char] = ["Person"]
            ents.add(str(ent).lower())
            
    doc = ''.join(doclist)
    return doc

def get_full_phrase(token):
    # Use noun_chunks for noun-based descriptions
    for chunk in token.doc.noun_chunks:
        if token in chunk:
            return chunk.text
    # Use adverb or adjective modifiers otherwise
    modifiers = [child for child in token.children if child.dep_ in {"advmod", "amod"}]
    return " ".join([mod.text for mod in sorted(modifiers, key=lambda t: t.i)] + [token.text])


def extract_pronoun_descriptions(text, pronoun):
    pronoun = pronoun.lower()
    descriptions = {f"{pronoun}_are": [], f"{pronoun}_do": []}

    for sent in doc.sents:
        for token in sent:
            if token.text.lower() == pronoun and token.dep_ in {"nsubj", "nsubjpass"}:
                verb = token.head
                # Case 1: Linking verb like "be" => state/trait
                if verb.lemma_ == "be":
                    for child in verb.children:
                        #print(f"\t\t{child}")
                        if child.dep_ in {"attr", "acomp", "oprd"}:
                            #print("HERE!!")
                            phrase = get_full_phrase(child)
                            
                            descriptions[f"{pronoun}_are"].append(phrase)
                # Case 2: Action verb => just the verb
                else:
                    descriptions[f"{pronoun}_do"].append(verb.lemma_)
    return descriptions

import spacy
spacy.require_gpu()
import pandas as pd
from charai_utils import *
from tqdm import tqdm
nlp = spacy.load("en_core_web_trf")

df = pd.read_parquet("english_5per.parquet")

res = []
for i, row in tqdm(df.iterrows(),total=len(df)):
    if len(row.greeting) > 40:
        text = row.greeting.replace("{{user}}", "you")
        text = text.replace("\n",". ")
        doc = nlp(mask(text,nlp))
        for pron in ['you','he','she','person','they']:
            desc_you = extract_pronoun_descriptions(doc, pron)
            for d in desc_you[f"{pron}_are"]:
                res.append([row.url,pron,'are',d])
            for d in desc_you[f"{pron}_do"]:
                res.append([row.url,pron,'do',d])
        

pd.DataFrame(res).to_parquet("power_result.parquet",index=False)


In [1]:
import spacy
import pandas as pd
from tqdm import tqdm
nlp = spacy.load("en_core_web_trf")
df = pd.read_parquet("/data/characterai/english_5per.parquet")

In [4]:

res = []
for i, row in tqdm(df.iloc[:10].iterrows(),total=len(df)):
    if len(row.greeting) > 40:
        text = row.greeting.replace("{{user}}", "you")
        text = text.replace("\n",". ")
        doc = nlp(mask(text,nlp))
        for pron in ['you','he','she','person','they']:
            desc_you = extract_pronoun_descriptions(doc, pron)
            print(desc_you)
            for d in desc_you[f"{pron}_are"]:
                res.append([row.url,pron,'are',d])
            for d in desc_you[f"{pron}_do"]:
                res.append([row.url,pron,'do',d])
        

pd.DataFrame(res).to_parquet("power_result.parquet",index=False)


  0%|                                                                                         | 2/2365436 [00:00<77:41:45,  8.46it/s]

{'you_are': ['so unbearable'], 'you_do': ['braid', 'disturb']}
{'he_are': ['old'], 'he_do': ['read', 'notice', 'stand', 'work']}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': []}
{'they_are': [], 'they_do': []}
{'you_are': [], 'you_do': ['come']}
{'he_are': ['alone', '21'], 'he_do': ['work']}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': []}
{'they_are': [], 'they_do': []}


  0%|                                                                                         | 4/2365436 [00:00<78:54:52,  8.33it/s]

{'you_are': ['boyfriend Person'], 'you_do': ['notice', 'comfort']}
{'he_are': ['slightly caring', 'really tired'], 'he_do': ['act', 'show', 'rest']}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': []}
{'they_are': [], 'they_do': []}
{'you_are': ['slightly jealous', 'jealous'], 'you_do': []}
{'he_are': [], 'he_do': ['model', 'notice', 'smirk']}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': []}
{'they_are': [], 'they_do': []}


  0%|                                                                                         | 5/2365436 [00:00<77:11:10,  8.51it/s]

{'you_are': ['quiet'], 'you_do': ['try', 'go']}
{'he_are': [], 'he_do': ['seem']}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': ['date']}
{'they_are': [], 'they_do': []}
{'you_are': [], 'you_do': ['protect']}
{'he_are': [], 'he_do': []}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': []}
{'they_are': [], 'they_do': []}


  0%|                                                                                         | 9/2365436 [00:00<67:55:04,  9.67it/s]

{'you_are': [], 'you_do': ['go', 'see', 'recognise', 'cheer']}
{'he_are': [], 'he_do': []}
{'she_are': [], 'she_do': ['look']}
{'person_are': [], 'person_do': []}
{'they_are': [], 'they_do': []}
{'you_are': [], 'you_do': []}
{'he_are': [], 'he_do': []}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': []}
{'they_are': [], 'they_do': []}
{'you_are': [], 'you_do': []}
{'he_are': [], 'he_do': []}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': []}
{'they_are': [], 'they_do': []}


  0%|                                                                                        | 10/2365436 [00:01<73:24:56,  8.95it/s]

{'you_are': ['a hybrid'], 'you_do': ['turn', 'step', 'whine', 'hide', 'notice']}
{'he_are': [], 'he_do': ['look']}
{'she_are': [], 'she_do': []}
{'person_are': [], 'person_do': ['come']}
{'they_are': [], 'they_do': []}


## Code to generate similarities for power, gender, and evaluation concepts

In [ ]:
from sentence_transformers import SentenceTransformer
from charai_utils import *
sentence_model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2", 
                                     trust_remote_code=True,
                                    cache_folder= "./st_cache")

import pandas as pd
docs = pd.read_csv("to_embed_tobeanalysis.csv").val.values.tolist()
print(len(docs))
embeddings = sentence_model.encode(docs, show_progress_bar=True)
for concept,s1,s2 in [
    ('power',['powerful', 'big','strong','potent','dominant'],['powerless', 'little','weak','impotent','feeble']),
    ('evaluation',['good', 'nice', 'positive', 'great', 'better', 'awesome'],['bad', 'awful', 'negative', 'terrible', 'worse', 'horrible']),
    ('gender', ['man', 'men', 'he', 'him', 'his', 'his', 'boy', 'boys', 'male', 'masculine'],['woman', 'women', 'she', 'her', 'her', 'hers', 'girl', 'girls', 'female', 'feminine'])]:
    gen_similarity(sentence_model, embeddings,concept,docs,s1,s2)

 
